# DASYNET: PneumoniaMNIST Classification

This notebook implements **DASYNET**—a custom Convolutional Neural Network (CNN) built from scratch to classify chest X-rays (normal vs. pneumonia) from the MedMNIST collection.

The goal is to match benchmark-level accuracy (typically ~90-95% for simple lightweight CNNs on this dataset) using a logical sequence of Convolution, Batch Normalization, and Max Pooling layers.

In [1]:
# Install medmnist if not already installed
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.9 MB/s eta 0:00:00


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
import medmnist
from medmnist import INFO, Evaluator
from torch.utils.data import DataLoader

print(f"MedMNIST v{medmnist.__version__} @ {medmnist.__path__}")

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Data Loading and Preprocessing
data_flag = 'pneumoniamnist'
download = True
BATCH_SIZE = 128

info = INFO[data_flag]
n_classes = len(info['label'])
DataClass = getattr(medmnist, info['python_class'])

# Define Transformations (MedMNIST standard is 28x28 grayscale)
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Load Datasets
train_dataset = DataClass(split='train', transform=data_transform, download=download)
val_dataset = DataClass(split='val', transform=data_transform, download=download)
test_dataset = DataClass(split='test', transform=data_transform, download=download)

# Create Data Loaders
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# Use non-shuffled loaders for metric computation with MedMNIST evaluator
train_eval_loader = DataLoader(dataset=train_dataset, batch_size=2 * BATCH_SIZE, shuffle=False)
val_loader = DataLoader(dataset=val_dataset, batch_size=2 * BATCH_SIZE, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=2 * BATCH_SIZE, shuffle=False)

print(f"Training set: {len(train_dataset)} images")
print(f"Test set: {len(test_dataset)} images")

# 3. DASYNET Architecture
class DASYNET(nn.Module):
    def __init__(self, in_channels=1, num_classes=n_classes):
        super(DASYNET, self).__init__()

        # Block 1 - "Scanner": 28x28 -> 14x14
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2 - "Scanner": 14x14 -> 7x7
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 3 - "Scanner": 7x7 -> 3x3
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.flatten = nn.Flatten()

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(64 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

model = DASYNET(in_channels=1, num_classes=n_classes).to(device)

# 4. Initialization, Optimizer, and Loss
EPOCHS = 35
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# 5. Model Training Loop
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device).squeeze().long()

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct += outputs.max(1)[1].eq(targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_acc = 100. * correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device).squeeze().long()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            val_correct += outputs.max(1)[1].eq(targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = 100. * val_correct / val_total
    scheduler.step(epoch_val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%")

# 6. Evaluation
def test(split):
    model.eval()
    y_score = torch.tensor([])

    if split == 'train':
        loader = train_eval_loader
    elif split == 'test':
        loader = test_loader
    else:
        raise ValueError("split must be 'train' or 'test'")

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            scores = F.softmax(outputs, dim=1)
            y_score = torch.cat((y_score, scores.cpu()), 0)

    metrics = medmnist.Evaluator(data_flag, split).evaluate(y_score.numpy())
    print(f'{split.capitalize()} Set AUC: {metrics[0]:.4f} | Accuracy: {metrics[1]:.4f}')

print("\n--- Final Benchmarking ---")
test('train')
test('test')

MedMNIST v3.0.2 @ ['/usr/local/lib/python3.12/dist-packages/medmnist']
Using device: cpu
Training set: 4708 images
Test set: 624 images
Epoch [1/35] Train Loss: 0.2499, Acc: 89.46% | Val Loss: 0.7644, Acc: 75.57%
Epoch [5/35] Train Loss: 0.0558, Acc: 97.96% | Val Loss: 0.0785, Acc: 96.76%
Epoch [10/35] Train Loss: 0.0123, Acc: 99.66% | Val Loss: 0.0879, Acc: 96.95%
Epoch [15/35] Train Loss: 0.0026, Acc: 99.98% | Val Loss: 0.1037, Acc: 97.14%
Epoch [20/35] Train Loss: 0.0018, Acc: 100.00% | Val Loss: 0.1067, Acc: 97.33%
Epoch [25/35] Train Loss: 0.0014, Acc: 100.00% | Val Loss: 0.1096, Acc: 97.14%
Epoch [30/35] Train Loss: 0.0013, Acc: 100.00% | Val Loss: 0.1097, Acc: 97.14%
Epoch [35/35] Train Loss: 0.0011, Acc: 100.00% | Val Loss: 0.1112, Acc: 97.14%

--- Final Benchmarking ---
Train Set AUC: 1.0000 | Accuracy: 1.0000
Test Set AUC: 0.9515 | Accuracy: 0.8558


In [9]:
# Save the trained model weights to disk
model_path = 'dasynet_pneumonia.pth'
torch.save(model.state_dict(), model_path)

print(f"Model successfully saved to {model_path}")
print("You can now find it in the folder icon (Files) on the left sidebar.")

Model successfully saved to dasynet_pneumonia.pth
You can now find it in the folder icon (Files) on the left sidebar.
